# Wavebit Tutorial 6
## Elliptical and circular incidence -- the `polarization` pair

Every tutorial so far described the incident polarization with **`psi`**, an angle. An angle
can only name a *linear* state, and a great deal of optics lives outside that set: circular
dichroism, waveplates, magneto-optics, chiral metasurfaces and any polarimetry at all need
the **relative phase** between the two field components, which no angle carries.

`polarization = (a_s, a_p)` -- a pair of complex E-field amplitudes -- is the general
spelling, and it *supersedes* `psi` rather than replacing it: every `psi` remains a pair, and
the two are checked against each other to the last bit. Beside it sits a second angle,
**`chi`**, which turns `psi` into the two coordinates a polarization state actually has --
the form to sweep and to optimize over.

This tutorial covers:

1. what `psi` reaches, and the one thing it cannot spell
2. the pair, and the exact `psi` it is equivalent to
3. what the pair's magnitude and its phase each do (one is discarded, one is not)
4. reading the transmitted state: a grating used as a waveplate
5. the demonstration that needs the phase -- a gyrotropic slab, against a closed form
6. four ways to get it wrong

The normative statement of all of this is [`../docs/conventions.md`](../docs/conventions.md),
"General (elliptical) polarization"; every rule below is pinned by
[`../tests/test_polarization.py`](../tests/test_polarization.py).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import wavebit


---

## 1. What `psi` reaches

`psi` is the angle of **E** from the p direction, in the plane transverse to **k**:
`psi = pi/2` is pure s (TE-like), `psi = 0` pure p (TM-like). Written as amplitudes that is

$$ (a_s,\; a_p) = (\sin\psi,\; \cos\psi) $$

and both entries are **real**. Two real amplitudes oscillate in step, so the physical field
`Re[E e^{-i omega t}]` sweeps back and forth along one line and returns. Every `psi` is a
line; the set of lines is all `psi` has.

Give the two components a relative phase and the tip of **E** stops retracing its path and
starts going around -- an ellipse, or a circle when the components are equal and a quarter
cycle apart. That is what the pair adds, and the picture below is the whole argument.

Two conventions fix the drawing, and both are stated in
[`../docs/conventions.md`](../docs/conventions.md): the time factor is `e^{-i omega t}`, and
at normal incidence with `phi = 0` the (s, p) basis maps to Cartesian as
`(E_x, E_y) = (-a_p, a_s)`.


In [ ]:
def e_trace(a_s, a_p, n=241):
    """The physical E-field tip over one optical cycle, in the (x, y) plane."""
    wt = np.linspace(0, 2 * np.pi, n)
    E = np.array([-a_p, a_s], dtype=complex)[:, None] * np.exp(-1j * wt)[None, :]
    return E.real                                     # rows: E_x, E_y


states = [('psi = 0.7  ->  (sin 0.7, cos 0.7)', (np.sin(0.7), np.cos(0.7)), 'tab:blue'),
          ('(1, -1j)/sqrt(2)  -- circular',     (1 / np.sqrt(2), -1j / np.sqrt(2)), 'tab:red'),
          ('(1, 0.5j)  -- elliptical',          (1, 0.5j), 'tab:green')]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.9))
for ax, (label, pair, c) in zip(axes, states):
    tr = e_trace(*pair) / np.hypot(*e_trace(*pair)).max()
    ax.plot(tr[0], tr[1], color=c, lw=2)
    ax.plot(tr[0, ::20], tr[1, ::20], 'o', color=c, ms=3)      # equal time steps
    ax.annotate('', xy=tr[:, 45], xytext=tr[:, 20],
                arrowprops=dict(arrowstyle='-|>', color=c, lw=2, mutation_scale=18))
    ax.set_title(label, fontsize=9)
    ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.25); ax.set_aspect('equal')
    ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5); ax.grid(alpha=.3)
    ax.set_xlabel('$E_x$')
axes[0].set_ylabel('$E_y$')
fig.suptitle('one optical cycle of the incident field, dots at equal time steps', fontsize=10)
plt.tight_layout(); plt.show()


The left panel is the entire reach of `psi`: a segment, whatever angle you choose. The other
two are ordinary incident states that no `psi` can express.

**This is not a plotting curiosity.** Section 5 measures a slab on which *every* `psi` returns
the same transmission, to nine digits, while the two circular pairs differ by as much as 20
percentage points.


---

## 2. The pair, and the `psi` it equals

`polarization=(a_s, a_p)` states the incident polarization as two complex **E-field
amplitudes**, in the same (s, p) basis as `psi` and as the returned amplitudes `r_s`, `r_p`,
`t_s`, `t_p`:

| pair | is |
| --- | --- |
| `(1, 0)` | pure s (TE-like), i.e. `psi = pi/2` |
| `(0, 1)` | pure p (TM-like), i.e. `psi = 0` |
| `(sin(v), cos(v))` | exactly `psi = v` |
| `(1, -1j)/sqrt(2)`, `(1, +1j)/sqrt(2)` | the two circular states |

The third row is the one that makes the pair an *extension* rather than a second convention,
so it is worth running rather than believing. Take a two-layer 2D structure at oblique,
off-plane incidence -- the general case, where s and p are fully coupled -- and solve it both
ways.


In [ ]:
# A raster holds relative permittivity: 4.0 is the n = 2 pixel, 2.25 the n = 1.5 one.
STRUCTURE_2D = np.array([[[4.00, 1.0, 2.25, 1.0],
                          [1.00, 9.0, 1.00, 4.0]]] * 2)    # 2 layers, relative permittivity
CFG_2D = dict(backend=0, top=1.0, bot=2.25, theta=0.3, phi=0.4, truncation_order=dict(x=2, y=1),
              wavelength=900.0, period=dict(x=700.0, y=700.0), thickness=[200.0, 100.0])

v = 0.7
via_psi = wavebit.rcwa(psi=v, structure=STRUCTURE_2D, **CFG_2D).solve()
via_pair = wavebit.rcwa(polarization=(np.sin(v), np.cos(v)),
                          structure=STRUCTURE_2D, **CFG_2D).solve()

for name in ('R', 'T', 'r_s', 'r_p', 't_s', 't_p'):
    gap = np.abs(np.asarray(getattr(via_psi.incident, name))
                 - np.asarray(getattr(via_pair.incident, name))).max()
    print(f'{name:4s}  max |psi - pair| over all orders = {gap:.1e}')


Bit-identical, on the whole result surface. The same holds on the conical path, on both
connecting algorithms, on all three backends and under back-side illumination
(`incidence_side='bottom'`) -- that is what `test_polarization.py` checks, so you do not have
to re-derive which spelling a script used.

`res` is the incidence you asked for. The `s_inc` / `p_inc` variants beside it are
the pure-s and pure-p solves that the coupled system produces for free; they are columns of
the same factorized problem and **do not depend on the pair at all**.

The near field comes along too -- once it is asked for: reconstruction is declared at
construction with `answers='field'`, because the per-layer state it walks is kept while the
solve runs or not at all. `solve_field` reconstructs from the same boundary match,
so the pair reaches the field cell without any separate spelling:


In [ ]:
RES = dict(sample_x=5, sample_y=4, sample_z=6)
_, cell_psi = wavebit.rcwa(psi=v, structure=STRUCTURE_2D, answers='field',
                           **CFG_2D).solve_field(**RES)
_, cell_pair = wavebit.rcwa(polarization=(np.sin(v), np.cos(v)), answers='field',
                            structure=STRUCTURE_2D, **CFG_2D).solve_field(**RES)
print('field cell shape      :', np.asarray(cell_psi).shape)
print('max |psi - pair| field : %.1e'
      % np.abs(np.asarray(cell_psi) - np.asarray(cell_pair)).max())


---

## 3. The magnitude is discarded, the phase is not

Two rules, and they are not symmetric:

- **The overall magnitude is not read.** The pair is normalized to unit incident flux on the
  way in, so `|a_s|^2 + |a_p|^2 = 1` is not required and writing a bigger pair does not send
  in more light. Efficiencies stay per unit incident power, amplitudes per unit incident
  E-field.
- **The overall phase is kept.** A global phase is a real, observable choice of time origin
  once you look at complex amplitudes, so it rides through to `r` and `t` untouched.

What is *not* discarded either way is the **relative** phase between `a_s` and `a_p` -- that
is the physics the pair exists to carry.


In [ ]:
kw = dict(structure=STRUCTURE_2D, **CFG_2D)

small = wavebit.rcwa(polarization=(0.6, 0.8j), **kw).solve()      # |a| = 1
large = wavebit.rcwa(polarization=(3.0, 4.0j), **kw).solve()      # |a| = 5, same state
print('scale x5   -> max dT   = %.1e' % np.abs(np.asarray(small.incident.T)
                                               - np.asarray(large.incident.T)).max())
print('           -> max dt_s = %.1e' % np.abs(np.asarray(small.incident.t_s)
                                               - np.asarray(large.incident.t_s)).max())

phase = np.exp(0.4j)
base = wavebit.rcwa(polarization=(1, -1j), **kw).solve()
turned = wavebit.rcwa(polarization=(phase, -1j * phase), **kw).solve()
print('\nglobal phase e^{0.4i}:')
print('  t_s(0,0) ratio = %s   (the phase itself: %s)'
      % (np.round(turned.order(0, 0).t_s / base.order(0, 0).t_s, 12), np.round(phase, 12)))
print('  max dT         = %.1e   (power cannot see it)'
      % np.abs(np.asarray(turned.incident.T) - np.asarray(base.incident.T)).max())


So `(3, 4j)` and `(0.6, 0.8j)` are one incidence, and there is no need to normalize a pair
before passing it -- write `(1, -1j)` if that is what you mean and skip the `sqrt(2)`.


---

## 4. Reading the transmitted state: a grating as a waveplate

A 1D grating at normal incidence is **form-birefringent**: the field along the grooves and the
field across them see different effective indices, so s and p leave with different phases. That
is a waveplate, and a waveplate is exactly the device whose input and output want to be
described by pairs.

The grating below is silicon, 700 nm pitch, 40% duty, 300 nm tall, at 1550 nm; the thickness
was chosen by sweeping it until the two eigen-polarizations came out a quarter cycle apart.
Its own retardance is a property of the structure, so read it from the two pure states first.

**Note the routing.** The structure is 1D, but the pure 1D TE/TM path is single-polarization by
construction, so a pair sends the solve to the **conical** path -- same physics, conical cost.
`grating_type == 1` is that decision, and it is automatic.


In [ ]:
WL, PERIOD, DUTY, HEIGHT, N_SI = 1550.0, 700.0, 0.4, 300.0, 3.48
EPS_SI = wavebit.eps_from_index(N_SI)     # 12.1104 -- silicon, as the solver takes it
row = np.where(np.arange(200) < int(200 * DUTY), EPS_SI, 1.0)
STRUCTURE_1D = np.array([[row]])          # one layer, varying in x, invariant in y

GRATING = dict(backend=0, wavelength=WL, period=[PERIOD, PERIOD], truncation_order=dict(x=15, y=0),
               top=1.0, bot=1.0, theta=0.0, phi=0.0, thickness=[HEIGHT])


def transmitted(pair):
    """The zeroth-order view of one incidence."""
    solver = wavebit.rcwa(polarization=pair, structure=STRUCTURE_1D, **GRATING)
    return solver, solver.solve().order(0, 0)


solver, pure_s = transmitted((1, 0))       # E along the grooves
_, pure_p = transmitted((0, 1))            # E across them

print('grating_type =', solver.grating_type, '(1 = conical)')
print(f't_s (pure s in) = {complex(pure_s.t_s):.4f}   T = {pure_s.T:.4f}')
print(f't_p (pure p in) = {complex(pure_p.t_p):.4f}   T = {pure_p.T:.4f}')
print(f'the plate itself: |t_p/t_s| = {abs(pure_p.t_p / pure_s.t_s):.4f}, '
      f'retardance = {np.degrees(np.angle(pure_p.t_p / pure_s.t_s)):.2f} deg')


A quarter cycle of retardance on nearly equal magnitudes: a quarter-wave plate, at 96-98%
transmission. A circular state arrives with its components a quarter cycle apart, so they
leave in step apart from a sign -- which is to say **linear**.

Reading that off a pair is worth a helper, since it is the same three numbers every time. The
polarization ellipse of `(a_s, a_p)` follows from the two conventions used in section 1: write
the Cartesian pair `(E_x, E_y) = (-a_p, a_s)`, rotate its global phase until the real and
imaginary parts are orthogonal, and those two vectors are the ellipse's axes -- with the field
running from the first to the second as time advances, because the time factor is
`e^{-i omega t}`.


In [ ]:
def ellipse_of(a_s, a_p):
    """(orientation in degrees, minor/major axis ratio, rotation sense) of the ellipse."""
    E = np.array([-a_p, a_s], dtype=complex)          # (E_x, E_y)
    F = E * np.exp(-0.5j * np.angle(E @ E))           # now Re(F) and Im(F) are orthogonal
    A, B = F.real, F.imag                             # E(t) = A cos(wt) + B sin(wt)
    if np.linalg.norm(B) > np.linalg.norm(A):
        A, B = B, -A                                  # keep A the major axis
    return (np.degrees(np.arctan2(A[1], A[0])) % 180,
            np.linalg.norm(B) / np.linalg.norm(A),
            np.sign(A[0] * B[1] - A[1] * B[0]))       # +1: x to y, -1: x to -y


# Optics names, read for light arriving from the top -- the default, and every solve in
# this tutorial. From the bottom the same two pairs swap names: see the note below.
RCP = (1 / np.sqrt(2), +1j / np.sqrt(2))
LCP = (1 / np.sqrt(2), -1j / np.sqrt(2))

for label, pair in (('incident RCP', RCP), ('incident LCP', LCP)):
    ang, ratio, sense = ellipse_of(*pair)
    print(f'{label:26s} axis ratio = {ratio:.4f}  sense = {sense:+.0f}')

for label, pair in (('transmitted from RCP', RCP), ('transmitted from LCP', LCP)):
    _, zero = transmitted(pair)
    ang, ratio, sense = ellipse_of(complex(zero.t_s), complex(zero.t_p))
    print(f'{label:26s} axis ratio = {ratio:.4f}  orientation = {ang:6.2f} deg'
          f'  T = {zero.T:.6f}')


Both circular inputs leave **linear** (axis ratio 0.001), and the two output lines are
**orthogonal**: 45.4 and 134.6 degrees. Put a linear polarizer behind this grating and you have
a circular-polarization analyzer -- one handedness passes, the other is blocked. None of it is
reachable through `psi`: the inputs are not linear states, and that the outputs are is the
measurement.

`sense` is the rotation about `+z`. RCP's `+1` turns the field from `+x` toward `+y`, a
rotation vector along `+z`, while the wave travels `-z`, down into the stack from the top:
the rotation is **antiparallel** to **k**, which is what optics calls right circular. The
table below has both states, from both sides.


In [ ]:
def out_pair(pair):
    zero = transmitted(pair)[1]
    return complex(zero.t_s), complex(zero.t_p)


panels = [('incident $(a_s, a_p)$ = RCP', RCP),
          ('transmitted from RCP', out_pair(RCP)),
          ('transmitted from LCP', out_pair(LCP))]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.9))
for ax, (label, pair) in zip(axes, panels):
    tr = e_trace(*pair)
    ax.plot(tr[0], tr[1], lw=2, color='tab:purple')
    ax.annotate('', xy=tr[:, 45], xytext=tr[:, 20],
                arrowprops=dict(arrowstyle='-|>', color='tab:purple', lw=2, mutation_scale=18))
    ax.set_title(label, fontsize=10); ax.set_aspect('equal')
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5); ax.grid(alpha=.3)
    ax.set_xlabel('$E_x$')
axes[0].set_ylabel('$E_y$')
plt.tight_layout(); plt.show()


### The names, and why the API does not use them

The two circular states have four names between them, and two of them are wrong depending on
who you ask: optics names handedness from the receiver's point of view, IEEE from the source's,
so **optics-RCP is IEEE-LHCP for the same physical field**. And a pair does not name a
handedness by itself: it is written in the fixed `(x, y)` frame, so the same pair turns the
other way *relative to its own propagation* when the light arrives from below. Wavebit
therefore takes the raw pair and never asks you which camp you are in. The rotation itself
is fixed by conventions this library already pins; `conventions.md` derives it, and
`test_polarization.py` measures it on the reconstructed field:

| pair | rotation vector | from the top (**k** down, the default) | from the bottom (**k** up) |
| --- | --- | --- | --- |
| `(1, -1j)/sqrt(2)` | `-z` | parallel: optics **LCP**, IEEE RHCP | antiparallel: optics **RCP**, IEEE LHCP |
| `(1, +1j)/sqrt(2)` | `+z` | antiparallel: optics **RCP**, IEEE LHCP | parallel: optics **LCP**, IEEE RHCP |

The `RCP` and `LCP` defined in section 4 are the top column. (Until 2026-09-21 `+z` pointed
down into the stack and top incidence travelled `+z`, so this table read the other way round
and so did those two names. The amplitudes did not change; the frame did.)

### One thing this grating cannot show you

Run the same grating with the other circular state and the powers do not move:


In [ ]:
for label, pair in (('RCP (1, +1j)', RCP), ('LCP (1, -1j)', LCP),
                    ('linear psi=0', (0, 1)), ('linear psi=pi/2', (1, 0))):
    print(f'{label:16s}  T(order 0) = {transmitted(pair)[1].T:.6f}')


The two circular states transmit **identically**, to six digits, while the two linear ones do
not. That is not a bug and not a sign the pair was ignored -- it is the structure. A straight
grating is achiral and mirror symmetric about its own plane of incidence, and such a structure
cannot tell the two handednesses apart in *power*. It acts on the *state* instead, which is why
the ellipses above moved and the efficiencies did not.

To see a handedness difference in the power itself you need a structure that has one, and that
is the next section.


---

## 5. The test the phase has to pass: a gyrotropic slab

A magneto-optic (Faraday) medium has an antisymmetric in-plane permittivity,

$$ \epsilon = \begin{pmatrix} \epsilon_d & i\gamma \\ -i\gamma & \epsilon_d \end{pmatrix} $$

whose eigenstates are **circular**, not linear. Each circular state sees a plain scalar slab
of index `sqrt(eps_d +/- gamma)` -- an independent Fabry-Perot with a closed-form Airy answer.
So this structure gives us all three things at once: the two circular pairs must differ, every
linear `psi` must give the same answer, and there is an exact number to check against.

A *patterned* anisotropic layer is not supported yet, but a **uniform** one has been solvable
all along by filling the five convolution slots directly
([`../docs/anisotropic-materials.md`](../docs/anisotropic-materials.md) section 4). Watch the
slot sign convention: the blocks hold `-eps_xy` and `-eps_yx`, not `+`.


In [ ]:
WL_M, EPS_D, GAMMA = 600.0, 2.25, 0.45


def gyrotropic(pair_or_psi, thickness, eps_d=EPS_D, gamma=GAMMA, wl=WL_M):
    """One uniform gyrotropic layer, driven through the five convolution slots."""
    I = np.eye(1, dtype=np.complex128)                     # truncation_order = dict(x=0, y=0): one order
    inc = (pair_or_psi if isinstance(pair_or_psi, dict)          # any spelling
           else {'polarization': pair_or_psi} if isinstance(pair_or_psi, tuple)
           else {'psi': float(pair_or_psi)})
    solver = wavebit.rcwa(backend=0, top=1.0, bot=1.0, theta=0.0, phi=0.0,
                            truncation_order=dict(x=0, y=0), period=dict(x=300.0, y=300.0), wavelength=wl,
                            thickness=[thickness], **inc)
    return solver.solve_conv(
        np.stack([eps_d * I]),                             # eps_xx
        np.stack([eps_d * I]),                             # eps_yy
        np.stack([np.linalg.inv(eps_d * I)]),              # 1 / eps_zz
        np.stack([-(+1j * gamma) * I]),                    # the slot holds -eps_xy
        np.stack([-(-1j * gamma) * I]),                    # ... and -eps_yx
    )


def airy(eps, thickness, wl=WL_M):
    """Normal-incidence (r, t) of a scalar slab in vacuum -- the textbook answer."""
    n1 = np.sqrt(eps + 0j)
    r01, t01 = (1 - n1) / (1 + n1), 2 / (1 + n1)
    r10, t10 = -r01, 2 * n1 / (1 + n1)
    ph = np.exp(2j * np.pi / wl * n1 * thickness)
    den = 1 + r01 * r10 * ph ** 2
    return r01 + t01 * t10 * r10 * ph ** 2 / den, t01 * t10 * ph / den


D = 220.0
print('every linear psi sees the same slab:')
for psi in np.linspace(0, np.pi / 2, 5):
    res = gyrotropic(psi, D)
    print(f'  psi = {psi:.4f}   T = {np.asarray(res.incident.T).sum():.9f}')

print('\nthe two circular states do not:')
for label, pair, sign in (('RCP (1, +1j)/sqrt(2)', RCP, -1), ('LCP (1, -1j)/sqrt(2)', LCP, +1)):
    res = gyrotropic(pair, D)
    t_ref = airy(EPS_D + sign * GAMMA, D)[1]
    print(f'  {label}   T = {np.asarray(res.incident.T).sum():.9f}'
          f'   Airy |t|^2 = {abs(t_ref) ** 2:.9f}'
          f'   (n = sqrt(eps_d {"+" if sign > 0 else "-"} gamma) = {np.sqrt(EPS_D + sign * GAMMA):.4f})')


Five values of `psi`, nine identical digits -- the linear states are blind to the gyrotropy
because each is an equal mixture of the two eigenstates. The circular pairs split by 9
percentage points at this thickness, and each lands on the Airy value for **its own**
eigen-index.

The amplitudes carry it too, relative phase included, which is the strongest form of the
statement: the transmitted pair is the incident pair times a scalar.


In [ ]:
res = gyrotropic(LCP, D)
t_ref = airy(EPS_D + GAMMA, D)[1]
print(f't_s  = {complex(res.incident.t_s.sum()):.12f}   Airy t * a_s = {t_ref * LCP[0]:.12f}')
print(f't_p  = {complex(res.incident.t_p.sum()):.12f}   Airy t * a_p = {t_ref * LCP[1]:.12f}')
print(f'the state is preserved: (t_s, t_p) / t  = '
      f'({complex(res.incident.t_s.sum()) / t_ref:.6f}, {complex(res.incident.t_p.sum()) / t_ref:.6f})')


In [ ]:
thick = np.arange(40.0, 601.0, 10.0)
T_rcp = np.array([np.asarray(gyrotropic(RCP, float(d)).incident.T).sum() for d in thick])
T_lcp = np.array([np.asarray(gyrotropic(LCP, float(d)).incident.T).sum() for d in thick])
T_lin = np.array([np.asarray(gyrotropic(0.7, float(d)).incident.T).sum() for d in thick])

plt.figure(figsize=(9, 4.6))
plt.plot(thick, [abs(airy(EPS_D + GAMMA, float(d))[1]) ** 2 for d in thick],
         '-', color='tab:red', lw=1, label='Airy, $n=\\sqrt{\\epsilon_d+\\gamma}$')
plt.plot(thick, [abs(airy(EPS_D - GAMMA, float(d))[1]) ** 2 for d in thick],
         '-', color='tab:blue', lw=1, label='Airy, $n=\\sqrt{\\epsilon_d-\\gamma}$')
plt.plot(thick[::2], T_lcp[::2], 'o', color='tab:red', ms=5, mfc='none',
         label='Wavebit, LCP = (1, -1j)/$\\sqrt{2}$')
plt.plot(thick[::2], T_rcp[::2], 's', color='tab:blue', ms=5, mfc='none',
         label='Wavebit, RCP = (1, +1j)/$\\sqrt{2}$')
plt.plot(thick, T_lin, '--', color='0.4', lw=1.2, label='any linear psi (their mean)')
plt.xlabel('slab thickness (nm)'); plt.ylabel('transmitted power')
plt.title('circular eigenstates of a gyrotropic slab: solver vs closed form')
plt.legend(fontsize=8, loc='lower left'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

print('max |Wavebit - Airy| over the sweep : %.1e'
      % max(np.abs(T_lcp - np.array([abs(airy(EPS_D + GAMMA, float(d))[1]) ** 2 for d in thick])).max(),
            np.abs(T_rcp - np.array([abs(airy(EPS_D - GAMMA, float(d))[1]) ** 2 for d in thick])).max()))
print('max |T(RCP) - T(LCP)|                 : %.3f' % np.abs(T_rcp - T_lcp).max())
print('max |T(linear) - mean of the two|     : %.1e'
      % np.abs(T_lin - 0.5 * (T_rcp + T_lcp)).max())


Machine precision against a closed form the solver knows nothing about, and a 20-point
separation between the two states that no `psi` can produce. The grey dashed line is not a
fit: a linear state is an equal mixture of the two circular eigenstates, so its transmission
is their **mean** -- which is what the curve is, to 1e-15, for every `psi`. Averaging away the
effect is precisely what linear incidence does here. This is
`test_polarization.py::test_each_circular_pair_is_one_faraday_eigenstate`, and it is the test
that the pair's *relative phase* reaches the boundary match with the meaning the documentation
assigns it -- nothing built out of linear incidence could have detected a wrong sign there.

**One scope limit, stated so it is not read past.** What is verified is that the solver
reproduces the tensor it is handed. Which physical magnetization direction the tensor
`[[eps_d, i*gamma], [-i*gamma, eps_d]]` denotes under this library's conventions is *not*
settled by any of this -- see [`../docs/anisotropic-materials.md`](../docs/anisotropic-materials.md)
section 6.1.


---

## 6. The two angles: `(psi, chi)`

The pair is the complete spelling, but it is not the convenient one to *move*. A polarization
state has exactly **two** degrees of freedom, and the pair spends four real numbers on them:
its overall magnitude is divided out and its overall phase cannot change any power, so two of
the four are directions that do nothing.

`chi` is the coordinate `psi` was always missing. `tan(chi)` is the ellipse's minor/major axis
ratio and its sign is the direction the field turns (`chi > 0` turns like `(1, -1j)`, which
makes `chi = +pi/4` the `LCP` of section 4), so:

| `chi` | state |
| --- | --- |
| `0` | linear -- exactly the `psi` spelling |
| `±pi/4` | circular |
| in between | elliptical, tilted at `psi` |

Together they are the standard coordinates of the state: `2*psi` and `2*chi` are longitude and
latitude on the Poincare sphere, so the two angles cover every state once, with nothing to
normalize and nothing redundant.

```
a_s = sin(psi)*cos(chi) + i*cos(psi)*sin(chi)
a_p = cos(psi)*cos(chi) - i*sin(psi)*sin(chi)
```


In [ ]:
# chi = 0 is the linear state at psi, and chi = pi/4 is circular -- checked against
# the pair each one names.
kw = dict(structure=STRUCTURE_2D, **CFG_2D)

same = [float(np.asarray(wavebit.rcwa(psi=0.7, **kw).solve().incident.T).sum()),
        float(np.asarray(wavebit.rcwa(psi=0.7, chi=0.0, **kw).solve().incident.T).sum())]
print(f'psi=0.7            T = {same[0]:.12f}')
print(f'psi=0.7, chi=0     T = {same[1]:.12f}      gap {abs(same[0] - same[1]):.1e}')

circular = float(np.asarray(
    wavebit.rcwa(psi=0.0, chi=np.pi / 4, **kw).solve().incident.T).sum())
by_pair = float(np.asarray(wavebit.rcwa(polarization=LCP, **kw).solve().incident.T).sum())
print(f'\npsi=0, chi=pi/4    T = {circular:.12f}')
print(f'polarization=LCP   T = {by_pair:.12f}      gap {abs(circular - by_pair):.1e}')


The gyrotropic slab from section 5 is the check that this is the *same* state and not merely
one with the same efficiency -- it is the one structure here that can tell the two circular
states apart.


In [ ]:
# The slab splits the two circular states; the chart has to land on the same two.
for label, pair, chi in (('LCP / chi = +pi/4', LCP, +np.pi / 4),
                         ('RCP / chi = -pi/4', RCP, -np.pi / 4)):
    t_pair = complex(gyrotropic(pair, D).incident.t_s.sum())
    t_chart = complex(gyrotropic({'psi': 0.0, 'chi': chi}, D).incident.t_s.sum())
    print(f'{label}:  |t| pair {abs(t_pair):.9f}   chart {abs(t_chart):.9f}   '
          f'gap {abs(abs(t_pair) - abs(t_chart)):.1e}')


The amplitudes agree in magnitude but not in phase, and that is the trade rather than a
defect: two coordinates carry two degrees of freedom, so the chart cannot also carry the
overall phase the pair does. Reach for `polarization` when that phase matters.

**Why have both, then?** Because the chart is the one to optimize over. Of the pair's four
real numbers, two cannot move the answer at all -- and a gradient says so:


In [ ]:
# The two coordinates carry gradients; the pair's other two directions are null.
import jax, jax.numpy as jnp

jkw = dict(kw, backend=1, structure=jnp.asarray(STRUCTURE_2D))


def total(**inc):
    return wavebit.rcwa(**inc, **jkw).solve().T.sum()


base = (jnp.sin(0.4), jnp.cos(0.4))
print('d(sum T) / d psi          = %+.6f' % jax.grad(lambda v: total(psi=v, chi=-0.2))(0.4))
print('d(sum T) / d chi          = %+.6f' % jax.grad(lambda v: total(psi=0.4, chi=v))(-0.2))
print('d(sum T) / d(pair scale)  = %+.1e   <- structurally null'
      % jax.grad(lambda t: total(polarization=tuple(t * x for x in base)))(1.0))
print('d(sum T) / d(pair phase)  = %+.1e   <- structurally null'
      % jax.grad(lambda u: total(polarization=tuple(jnp.exp(1j * u) * x for x in base)))(0.0))


So an optimizer handed the raw pair spends half its dimensions on directions that cannot
change the objective, while `(psi, chi)` is an ordinary unconstrained 2D problem.

One rule to remember: **naming `chi` selects the chart, including `chi = 0`.** Leaving it out
keeps the plain `psi` path, which applies no normalization because it has nothing to
normalize; the chart goes through the pair's. Same physics, and they can differ in the last
bit.


---

## 7. Five ways to get it wrong

### 7.1 Scaling the pair to get a brighter answer

The pair is normalized to unit incident flux, so a bigger pair is the same incidence.


In [ ]:
for pair in [(1, 0), (10, 0), (1e-6, 0)]:
    r = wavebit.rcwa(polarization=pair, structure=STRUCTURE_2D, **CFG_2D).solve()
    print(f'polarization={str(pair):12s} ->  sum T = {np.asarray(r.incident.T).sum():.12f}')


Nothing about the illumination changed, and nothing warns, because nothing is wrong -- `R`,
`T` are fractions of the incident power and `r`, `t` are ratios to the incident field,
whatever you write. If you want absolute power, multiply the efficiencies by it afterwards.

The one pair that *is* refused is the empty one:


In [ ]:
for bad in [(0, 0), (1, 0, 0)]:
    try:
        wavebit.rcwa(backend=0, polarization=bad)
    except ValueError as exc:
        print(f'polarization={bad!s:10s} -> ValueError: {exc}')


### 7.2 Passing `psi` and `polarization` together

They specify the same thing, so naming both is an **error, not an override** -- in either
order, and whether at construction or by assignment. A silent winner would be far worse: the
loser is a polarization you meant.


In [ ]:
try:
    wavebit.rcwa(backend=0, psi=0.3, polarization=(1, 0))
except ValueError as exc:
    print('at construction ->', exc)

solver = wavebit.rcwa(backend=0, polarization=(1, 1j))
try:
    solver.psi = 0.3
except ValueError as exc:
    print('\nby assignment   ->', exc)

solver.polarization = None            # clearing one reopens the other spelling
solver.psi = 0.3
print('\nafter clearing the pair, psi =', solver.psi)

# `tetm` declares the polarization away, so it refuses a pair rather than losing to it
s2 = wavebit.rcwa(backend=0, tetm='tm')
try:
    s2.polarization = (1, 0)
except ValueError as exc:
    print('tetm then a pair ->', exc)


### 7.3 Taking a Jones vector from an `e^{+i omega t}` source verbatim

Wavebit uses `e^{-i omega t}` (`Im(n) > 0` is loss). A pair quoted from a source using the
other time convention -- older papers, engineering texts, anything writing `n - jk` -- is the
**complex conjugate** of ours, and conjugating a Jones vector swaps the two circular states.
Nothing warns: the wrong state is a perfectly good state.


In [ ]:
their_rcp = (1 / np.sqrt(2), -1j / np.sqrt(2))       # "RCP" as an e^{+i omega t} source writes it
ours = tuple(np.conj(their_rcp))                     # the same physical field, our convention: RCP

T = {}
for label, pair in (('used verbatim (wrong)', their_rcp), ('conjugated  (right)', ours)):
    res = gyrotropic(pair, D)
    T[label] = float(np.asarray(res.incident.T).sum())
    r_plus_t = T[label] + float(np.asarray(res.incident.R).sum())
    print(f'{label:24s} T = {T[label]:.9f}    R + T = {r_plus_t:.9f}')
print(f'\nthe two answers differ by {abs(T["used verbatim (wrong)"] - T["conjugated  (right)"]):.4f}'
      ', and energy conserves either way -- nothing flags it')


The same seam runs the other direction, on the way out: `r_s`, `r_p`, `t_s`, `t_p` are in our
convention, so comparing them against amplitudes from an `e^{+i omega t}` source means
conjugating one side. Powers are unaffected -- `|t|^2` does not care -- which is exactly why
this survives a check that only looks at efficiencies.

### 7.4 Expecting a handedness contrast from a structure that has none

Section 4 is the cautionary case: on the straight grating both circular states transmitted
identically to six digits, and it would be easy to conclude the pair "does nothing". Power
distinguishes handedness only where the structure does -- chirality, or a magneto-optic
tensor, or at least an incidence that breaks the mirror symmetry. The *state* moved on that
grating even so, and the amplitudes showed it. When a circular-dichroism number comes out at
zero, check the symmetry before checking the solver.


### 7.5 Reaching for a complex `psi`

If a relative phase is what the pair adds, an imaginary part on the angle looks like the
cheap way to get one. It is refused, and it would have been wrong twice over: `(sin, cos)`
stops being a unit pair, so the incident flux silently becomes `cosh(2*Im(psi))` and every
efficiency carries that factor -- and it cannot reach the circular states anyway, because
`a_s/a_p = tan(psi)` and `tan` never takes the values `±i`.


In [ ]:
try:
    wavebit.rcwa(backend=0, psi=0.7 + 0.5j)
except ValueError as exc:
    print(str(exc)[:220], '...')


---

## 8. Summary

| you want | write |
| --- | --- |
| linear at angle `v` | `psi=v`, or `polarization=(sin(v), cos(v))` -- identical |
| pure s / pure p | `polarization=(1, 0)` / `(0, 1)` |
| circular | `polarization=(1, -1j)` / `(1, 1j)`, or `chi=±pi/4` -- no need to normalize |
| elliptical, any orientation | `polarization=(a_s, a_p)`, any two complex numbers |
| to sweep or optimize the state | `psi` and `chi` -- two real angles, no null directions |
| the overall phase to survive | `polarization`; the chart does not carry it |
| a 1D structure, non-linear state | the same call: the conical path is selected for you |
| the fast 1D path, giving up both | `tetm='te'` / `'tm'` -- a declaration, not an angle |

And the rules behind them, each pinned by
[`../tests/test_polarization.py`](../tests/test_polarization.py):

1. `psi = v` **is** `polarization = (sin(v), cos(v))` -- not an approximation of it.
2. The pair's magnitude is normalized away (unit incident flux); its overall phase is kept.
3. `psi` and `polarization` together raise, in either order -- and so do `chi` with the pair,
   and `tetm` with any of them.
4. A 1D structure given a pair is solved on the conical path.
5. `chi = 0` is the linear state at `psi`, `chi = ±pi/4` the circular ones; naming `chi` at
   all selects the chart.

The normative statement is [`../docs/conventions.md`](../docs/conventions.md) -- "General
(elliptical) polarization" for the rules, "Which way the circular states rotate" for the
handedness derivation and why the API refuses to name the states itself.
